# Reasoning Agent

In [ ]:
!ollama pull llama3.2:3b

In [ ]:
!ollama pull deepseek-r1:7b

In [ ]:
!ollama pull qwen3.5

In [ ]:
!ollama pull qwen3:0.6b # this model fails to call the tools correctly

In [8]:
PLANNER_MODEL = 'deepseek-r1:7b' # reasoning model (our planner)
EXECUTOR_MODEL = 'llama3.2:3b' # vanilla LLM (our plan executer)
#EXECUTOR_MODEL = 'qwen3:0.6b'
#EXECUTOR_MODEL= 'glm-5.1:cloud'
#EXECUTOR_MODEL = 'qwen3.5'

## Tools

In [ ]:
folder = './profile/'
def read_profile() -> str:
    """
    Reads all profile files returns them as one text.
    """
    filenames = ["personal_profile.txt", "work_profile.txt", "communication_profile.txt"]
    aggregation = ""
    for file in filenames:
      open(folder + file, "a").close() # create if doesn't exist

      with open(folder + file, "r") as f:
          aggregation += f.read()
    return aggregation

def add_to_personal_profile(fact: str):
    """Store one personal fact about the user.

    Args:
        fact: The fact as a plain English sentence, e.g. "lives in Switzerland"
    """
    with open(folder + "personal_profile.txt", "a") as f:
      f.write(f"{fact};\n")

def add_to_work_profile(fact: str):
    """Store one fact about what the user is working on.

    Args:
        fact: The fact as a plain English sentence, e.g. "is working on a web application"
    """
    with open(folder + "work_profile.txt", "a") as f:
      f.write(f"{fact};\n")


def add_to_communication_profile(fact: str):
    """Store one communication fact about the user.

    Args:
        fact: The fact as a plain English sentence, e.g. "communicates directly"
    """
    with open(folder + "communication_profile.txt", "a") as f:
      f.write(f"{fact};\n")

### Specific Use Case Tools

In [ ]:
import requests
import geoip2.database

def get_user_location():
    add_to_personal_profile(get_geo_locally(get_public_ip()))

def get_public_ip():
    try:
        response = requests.get('https://api.ipify.org?format=json')
        return response.json()['ip']
    except requests.RequestException as e:
        return f"Error: {e}"

def get_geo_locally(ip_address):
    reader = geoip2.database.Reader('./GeoLite2-City.mmdb')
    try:
        response = reader.city(ip_address)
        return f"country={response.country.name}, state={response.subdivisions.most_specific.name}, city={response.city.name}, latitude={response.location.latitude}, longitude={response.location.longitude}"
    except Exception as e:
        return f"Error: {e}"
    finally:
        reader.close()

print(get_geo_locally(get_public_ip()))

### Helper Functions

In [11]:
prompts_file = "user_prompts.txt"
def grab_prompts_from_user():
    open(prompts_file, "a").close() # create if doesn't exist
    with open(prompts_file, "r") as f:
        return f.read()

In [ ]:
import ollama
import re

user_prompt = grab_prompts_from_user()
current_knowledge = read_profile()

scenario = """
This is a user behavior analysis system. The smarter model is the planner who plans how to analyze the user and there is a less complex model that is the executor
that is executing the plan with the tools given. Each user adheres strictly to their role and fulfills their role to the best of their abilities.

The user is interacting with a system that analyses the user based on their prompts.
to gather information on the user. This data is stored as semicolon separated facts into multiple category files. These files are split into "personal", "communication_style".
The personal category is about who the user is and what they are about (e.g. name, age, location, interests).
The work category is about what the user does (e.g. summarizing text, creating images, projects)
The communication category is about how the user communicates (e.g. arrogant, direct, polite).

The system has these functions: 'read_profile', 'add_to_personal_profile',  'add_to_work_profile' and 'add_to_communication_style_profile'.
"""

planner_prompt = f"""
You are a behavioral analysis planner. Your job is to analyze a user's query and produce an EXECUTION PLAN that a simple tool-calling model will follow literally.

## WHAT TO EXTRACT

Analyze the user query below and extract facts in these three categories:

**PERSONAL** (who the user is):
- Name, age, gender, nationality, location
- Education level, field of study
- Hobbies, interests, preferences
- Languages spoken
- Life circumstances mentioned
- Location with get_user_location() tool call

**WORK** (what the user does):
- Profession or role
- Projects, tasks, goals mentioned
- Tools, technologies, frameworks referenced
- Domain expertise implied by terminology

**COMMUNICATION** (how the user writes):
- Formality level (casual, neutral, formal)
- Vocabulary complexity (simple, technical, academic)
- Emotional tone (frustrated, curious, confident, uncertain)
- Sentence structure patterns (short/direct, long/elaborate)
- Politeness markers, slang, or jargon

## EXISTING PROFILE
{current_knowledge if current_knowledge else "Empty, no data yet."}

Do NOT store facts that are already in the existing profile.
Extract as much information as possible.

## OUTPUT FORMAT

You MUST output your plan as a numbered list of CALL lines. Each line is ONE function call. Use ONLY these functions:

- add_to_personal_profile(fact)
- add_to_work_profile(fact)
- add_to_communication_profile(fact)
- read_profile()
- get_user_location()

Format each step EXACTLY like this:
STEP 1: CALL add_to_personal_profile("the extracted fact here")
STEP 2: CALL add_to_work_profile("another fact here")

Rules:
- One fact per CALL. Do not combine multiple facts into one call.
- Facts must be plain English. No JSON, no special formatting.
- Extract as much information as possible.
- Find and store at least 5 facts for every category
- Find at least 20 facts in the query
- Start by calling the function get_user_location() without parameters
- End with STEP N: DONE

## USER QUERY
<user-query>
{user_prompt}
</user-query>

Now produce the execution plan.
"""
r1_response = ollama.chat(
    PLANNER_MODEL,
    messages=[{'role': 'user', 'content': planner_prompt}],
)

raw_plan = r1_response.message.content
print(f"Raw Plan: {raw_plan}")
plan_cleaned = re.sub(r'<think>.*?</think>', '', raw_plan, flags=re.DOTALL).strip()

executor_prompt = f"""
You are a tool-calling executor. You receive a numbered plan and execute each STEP by calling the corresponding tool.

RULES:
- Execute every STEP in order. Do not skip any.
- Each CALL line maps directly to one tool call.
- Use the function name and argument exactly as written in the plan.
- When you reach DONE, stop.
- Do not add any extra tool calls beyond what is stated says.
- Your only job is to execute the plan.

PLAN:
{plan_cleaned}
"""

available_functions = {
    'read_profile': read_profile,
    'add_to_personal_profile': add_to_personal_profile,
    'add_to_communication_profile': add_to_communication_profile,
    'add_to_work_profile': add_to_work_profile,
    'get_user_location': get_user_location,
}

messages = [{'role': 'user', 'content': executor_prompt}]

while True:
    response = ollama.chat(
        EXECUTOR_MODEL,
        messages=messages,
        tools=[read_profile, add_to_personal_profile, add_to_communication_profile, add_to_work_profile, get_user_location],
    )

    if not response.message.tool_calls:
        break

    messages.append(response.message)

    if response.message.tool_calls:
        for tool in response.message.tool_calls:
            func = available_functions.get(tool.function.name)
            if func:
                try:
                    args = tool.function.arguments
                    if isinstance(args, str):
                        args = {"fact": args} 
                    
                    result = func(**args)
                    messages.append({
                        'role': 'tool',
                        'content': str(result) if result else 'Done.',
                    })
                except TypeError as e:
                    error_msg = f"Error: Failed to call {tool.function.name}. {e}. Please correct your parameters and try again."
                    print(f"Model fumbled: {error_msg}")
                    messages.append({
                        'role': 'tool',
                        'content': error_msg,
                    })
                except Exception as e:
                    messages.append({
                        'role': 'tool',
                        'content': f"Error executing tool: {e}",
                    })
            else:
                messages.append({
                    'role': 'tool',
                    'content': f'Error: unknown function {tool.function.name}',
                })

## Compression

In [24]:
import ollama
import os

folder = './profile/'
for file in [folder + "personal_profile.txt", folder + "work_profile.txt", folder + "communication_profile.txt"]:
    open(file, "a").close() # create if doesn't exist
    content = ""
    with open(file, "r") as f:
        content = f.read()
    
    summarizer_prompt = f"""
    You are an expert summarizer. You extract and distill the important parts from the text given and summarize it.
    The summary focuses on extracting information about the user who wrote this. Extract curcial information and build a profile
    about the given information. When summarizing extract the bigger picture and feel free to leave away unnecessary details.
    Separate each fact that you can extract into separate lines that are terminated with a semicolon ';'.

    <content-to-be-summarized>
    {content}
    </content-to-be-summarized>
    """

    r1_response = ollama.chat(
        PLANNER_MODEL,
        messages=[{'role': 'user', 'content': summarizer_prompt}],
    )

    os.remove(file)
    with open(file, "a") as f:
      f.write(r1_response.message.content)